[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alsymiya/Argentina/blob/main/1.%20draw_demo%20%28Colab%29.ipynb)

Click the badge to open this notebook directly in Google Colab. Then run the Setup cell - it will download the helper .py files from [alsymiya/Argentina](https://github.com/alsymiya/Argentina) automatically. No manual file copying needed.

# Draw a curve and watch it get normalized

Freehand-draw any curve in the canvas below, then run the next cell to see it go through the four-stage geometric normalization pipeline from `normalize.py`:

1. **center** - subtract the centroid
2. **scale** - rescale so the max distance from the origin equals 3.5
3. **rotate_pca** - align the greatest principal axis with +X
4. **reflect_moments** - sign-flip via third-order moments so the canonical form is unique

Every congruent/scaled/mirrored version of the same intrinsic shape lands in the same canonical pose. That's exactly the preprocessing the MotionGen path-synthesis backend uses before it does a KDTree nearest-neighbour lookup.

## Setup

Run the next cell once. On Colab it locates the Argentina helper files in your Google Drive. The freehand canvas uses a plain HTML5 `<canvas>` (no `ipympl`, no kernel restart needed).

In [ ]:
import os, sys, urllib.request
print(f'kernel PID: {os.getpid()}')

IN_COLAB = 'google.colab' in sys.modules
HELPERS = ['simulator.py', 'optimizer.py', 'visualizer.py', 'motiongen_export.py', 'normalize.py', 'bsi_converter.py', 'example_config.txt']
REPO_BASE = 'https://raw.githubusercontent.com/alsymiya/Argentina/main/'

if not any(os.path.isfile(h) for h in HELPERS):
    # 1) Try GitHub raw download (no Drive needed)
    downloaded, failed = [], []
    for h in HELPERS:
        try:
            urllib.request.urlretrieve(REPO_BASE + urllib.parse.quote(h), h)
            downloaded.append(h)
        except Exception as e:
            failed.append((h, str(e)))
    if downloaded:
        print('downloaded from GitHub:', downloaded)
    if failed:
        print('failed from GitHub:')
        for h, err in failed:
            print(f'  {h}: {err}')

    # 2) If GitHub didn't deliver, try Drive (Colab only)
    if IN_COLAB and not any(os.path.isfile(h) for h in HELPERS):
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            print('Mounting Google Drive (auth prompt) ...')
            drive.mount('/content/drive')
        for c in ['/content/drive/MyDrive/Argentina',
                  '/content/drive/My Drive/Argentina']:
            if any(os.path.isfile(os.path.join(c, h)) for h in HELPERS):
                os.chdir(c)
                if c not in sys.path:
                    sys.path.insert(0, c)
                print(f'workdir -> {c}')
                break
        else:
            print('WARNING: helpers not found via GitHub OR Drive.')
            print('         Check your network or upload .py files to /content.')

import urllib.parse  # used above; safe to import here too

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from normalize import normalize, apply_transform, resample_polyline

# Shared state for the drawing canvas
_state = {'points': [], 'points_px': []}

# --- HTML5 freehand canvas (Colab-native; no ipympl) --------------------
CANVAS_SIZE = 500
CANVAS_XLIM = (-5.0, 5.0)
CANVAS_YLIM = (-5.0, 5.0)

def _px_to_data(points_px):
    """Map canvas pixels (origin top-left, y down) to data coordinates."""
    pts = np.asarray(points_px, dtype=float)
    if pts.size == 0:
        return pts.reshape(0, 2)
    x = pts[:, 0] / CANVAS_SIZE * (CANVAS_XLIM[1] - CANVAS_XLIM[0]) + CANVAS_XLIM[0]
    y = CANVAS_YLIM[1] - pts[:, 1] / CANVAS_SIZE * (CANVAS_YLIM[1] - CANVAS_YLIM[0])
    return np.column_stack([x, y])

def get_drawn_curve():
    """Most recent freehand stroke as an (N, 2) ndarray in data coords."""
    return np.array(_state['points'], dtype=float) if _state['points'] else None

_CANVAS_HTML = """
<div>
  <canvas id="drawcanvas" width="500" height="500"
          style="border:1px solid #888; cursor:crosshair; touch-action:none;"></canvas>
  <div style="margin-top:6px; font-family:monospace;">
    <button id="clearbtn">Clear</button>
    <span id="status" style="margin-left:10px;">0 points</span>
  </div>
</div>
<script>
(function() {
  const canvas = document.getElementById('drawcanvas');
  const ctx = canvas.getContext('2d');
  const status = document.getElementById('status');
  let drawing = false, points = [];
  function grid() {
    ctx.clearRect(0,0,canvas.width,canvas.height);
    ctx.strokeStyle='#eee'; ctx.lineWidth=1;
    for (let i=0;i<=10;i++){let p=i/10*canvas.width;
      ctx.beginPath();ctx.moveTo(p,0);ctx.lineTo(p,canvas.height);ctx.stroke();
      ctx.beginPath();ctx.moveTo(0,p);ctx.lineTo(canvas.width,p);ctx.stroke();}
    ctx.strokeStyle='#bbb';
    ctx.beginPath();ctx.moveTo(canvas.width/2,0);ctx.lineTo(canvas.width/2,canvas.height);ctx.stroke();
    ctx.beginPath();ctx.moveTo(0,canvas.height/2);ctx.lineTo(canvas.width,canvas.height/2);ctx.stroke();
  }
  grid();
  function pos(e){const r=canvas.getBoundingClientRect();return [e.clientX-r.left, e.clientY-r.top];}
  function send(){if(window.google&&google.colab&&google.colab.kernel){
      google.colab.kernel.invokeFunction('notebook.set_points',[points],{});}}
  canvas.addEventListener('mousedown',e=>{drawing=true;points=[pos(e)];grid();
    ctx.strokeStyle='#3366cc';ctx.lineWidth=2;ctx.beginPath();ctx.moveTo(points[0][0],points[0][1]);});
  canvas.addEventListener('mousemove',e=>{if(!drawing)return;const p=pos(e);points.push(p);
    ctx.lineTo(p[0],p[1]);ctx.stroke();status.textContent=points.length+' points';});
  function finish(){if(!drawing)return;drawing=false;
    status.textContent=points.length+' points (sent to Python)';send();}
  canvas.addEventListener('mouseup',finish);
  canvas.addEventListener('mouseleave',finish);
  document.getElementById('clearbtn').addEventListener('click',()=>{
    points=[];grid();status.textContent='0 points';send();});
})();
</script>
"""

def _register_canvas_callback():
    if not IN_COLAB:
        print('Note: the HTML5 canvas callback only works on Colab.')
        print('On local Jupyter, set _state["points"] = [[x,y], ...] manually.')
        return
    from google.colab import output as _o
    def _cb(points_px):
        _state['points_px'] = list(points_px)
        _state['points'] = _px_to_data(points_px).tolist()
    _o.register_callback('notebook.set_points', _cb)

def show_draw_canvas():
    """Render the freehand canvas. The stroke lands in _state['points']
    (data coords) when you release the mouse."""
    _state['points'] = []
    _state['points_px'] = []
    _register_canvas_callback()
    display(HTML(_CANVAS_HTML))
    print('Draw in the canvas above (click + drag, release to finish), then run the next cell.')


## Step 1 - Freehand canvas

Click and drag to draw; release to finish. Hit **Clear** to start over. The stroke is sent to Python automatically on release.

In [ ]:
show_draw_canvas()

## Step 2 - See the normalization pipeline

Five panels: your raw input, then each successive stage. The dashed circle in the *scaled* panel marks `max-norm = 3.5`; the red arrow in the *rotated* panel is the dominant principal axis.

In [ ]:
curve = get_drawn_curve()
assert curve is not None and curve.shape[0] >= 5, 'Please draw something first (at least 5 points).'

if curve.shape[0] > 500:
    curve = resample_polyline(curve, 500)

nc, M, success, stages = normalize(curve, scaling=3.5, return_stages=True)
print(f'success: {success}   |det(M)| = {abs(np.linalg.det(M)):.4f}')

def _square_lim(pts, pad=0.15):
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    c = 0.5 * (lo + hi)
    half = 0.5 * (hi - lo).max() * (1 + pad)
    half = max(half, 1e-3)
    return c[0] - half, c[0] + half, c[1] - half, c[1] + half

order = ['input', 'centered', 'scaled', 'rotated', 'reflected']
titles = {'input':'1. Input (as drawn)','centered':'2. Centered',
          'scaled':'3. Scaled (max-norm = 3.5)','rotated':'4. PCA-rotated',
          'reflected':'5. Reflected (canonical)'}

fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax_, name in zip(axes, order):
    c, T = stages[name]
    ax_.plot(c[:, 0], c[:, 1], '-', color='#3366cc', lw=2)
    ax_.plot(c[0, 0], c[0, 1], 'o', color='#22aa22', ms=6, label='start')
    ax_.plot(c[-1, 0], c[-1, 1], 'o', color='#cc3333', ms=6, label='end')
    xl, xh, yl, yh = _square_lim(c)
    ax_.set_xlim(xl, xh); ax_.set_ylim(yl, yh)
    ax_.set_aspect('equal'); ax_.grid(True, alpha=0.25)
    ax_.set_title(titles[name])

theta = np.linspace(0, 2*np.pi, 200)
axes[2].plot(3.5*np.cos(theta), 3.5*np.sin(theta), '--', color='gray', lw=1)
rot = stages['rotated'][0]
axes[3].annotate('', xy=(rot[:,0].max()*0.9, 0), xytext=(rot[:,0].min()*0.9, 0),
                 arrowprops=dict(arrowstyle='->', color='#cc3333', lw=2))
axes[0].legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

## Step 3 - Verify the composite transform

`apply_transform(curve, M)` should reproduce the canonical curve (up to float precision). This is the same `M` you'd ship to the MotionGen backend - the inverse-normalization parameter that maps a database mechanism back onto the user's drawn coordinate frame.

In [ ]:
recon = apply_transform(curve, M)
err = float(np.max(np.abs(recon - nc)))
print(f'max |apply_transform(curve, M) - nc| = {err:.2e}')

print('\nComposite transform M (3x3):')
with np.printoptions(precision=4, suppress=True):
    print(np.asarray(M))

print('\nPer-stage transforms:')
for name in ['centered', 'scaled', 'rotated', 'reflected']:
    _, T = stages[name]
    print(f'  {name:10s} det = {np.linalg.det(T):+.4f}')

## Step 4 - (Optional) Resample to a fixed length

`resample_polyline(nc, n)` gives you exactly `n` equispaced (by arclength) points - handy if a downstream encoder wants a fixed-length sequence.

In [ ]:
N = 64
resampled = resample_polyline(nc, N)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(nc[:, 0], nc[:, 1], '-', color='#3366cc', lw=1.5, alpha=0.5, label='canonical')
ax.plot(resampled[:, 0], resampled[:, 1], 'o', color='#cc3333', ms=4, label=f'resampled (N={N})')
ax.set_aspect('equal'); ax.grid(True, alpha=0.25); ax.legend(loc='upper right')
ax.set_title('Canonical curve, resampled by arclength')
plt.show()

---

### Why this is *not* the same as `normalize_curve` in `optimizer.py`

The optimizer's `normalize_curve` is a six-line *bounding-box* rescaling built from torch primitives. It lives inside the L-BFGS forward pass and the gradient of the loss has to flow through it back to the joint positions - so it has to be differentiable.

This module is *pose-canonicalization*. The PCA eigendecomposition and the sign-of-moments reflection are not (usefully) differentiable. That's fine here because the input is a fixed user-drawn curve. Don't drop this into the optimizer's forward pass.

### Local Jupyter note

The HTML5 canvas uses `google.colab.output`, which only exists on Colab. On a local kernel, `show_draw_canvas()` renders the canvas but the stroke can't round-trip back to Python - set `_state['points'] = [[x, y], ...]` manually, or use the older matplotlib-widget approach.